In [9]:
import pandas as pd

In [10]:
price_dataset = pd.read_csv("../data/raw/entso-e-prices/Sweden.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [11]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Sweden,SWE,2015-01-01 00:00:00,2015-01-01 01:00:00,23.37,2015-01-01 00:00:00
1,Sweden,SWE,2015-01-01 01:00:00,2015-01-01 02:00:00,19.33,2015-01-01 01:00:00
2,Sweden,SWE,2015-01-01 02:00:00,2015-01-01 03:00:00,17.66,2015-01-01 02:00:00
3,Sweden,SWE,2015-01-01 03:00:00,2015-01-01 04:00:00,17.53,2015-01-01 03:00:00
4,Sweden,SWE,2015-01-01 04:00:00,2015-01-01 05:00:00,18.07,2015-01-01 04:00:00


In [12]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [13]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,23.37,2015-01-01 00:00:00
1,19.33,2015-01-01 01:00:00
2,17.66,2015-01-01 02:00:00
3,17.53,2015-01-01 03:00:00
4,18.07,2015-01-01 04:00:00


In [14]:
solar_dataset = pd.read_csv("../data/raw/SE/solar-raw.csv")

In [15]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [16]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [17]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [18]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [19]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [20]:
meteo_dataset = pd.read_csv("../data/raw/meteo-raw-sweden.csv")

In [21]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,-0.6,8.0,333,100,0.0,0.0
1,2022-01-01T01:00,-0.9,9.6,326,100,0.0,0.4
2,2022-01-01T02:00,-0.7,12.9,330,100,0.0,0.3
3,2022-01-01T03:00,-0.2,14.4,1,100,0.0,0.0
4,2022-01-01T04:00,-0.4,12.2,360,100,0.0,0.0


In [22]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [23]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [24]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [25]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,-0.6,8.0,333,100,0.0,0.0,2022-01-01 00:00:00
1,-0.9,9.6,326,100,0.0,0.4,2022-01-01 01:00:00
2,-0.7,12.9,330,100,0.0,0.3,2022-01-01 02:00:00
3,-0.2,14.4,1,100,0.0,0.0,2022-01-01 03:00:00
4,-0.4,12.2,360,100,0.0,0.0,2022-01-01 04:00:00


In [26]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,41.33,2022-01-01 00:00:00,0.0,-0.6,8.0,333,100,0.0,0.0
1,42.18,2022-01-01 01:00:00,0.0,-0.9,9.6,326,100,0.0,0.4
2,44.37,2022-01-01 02:00:00,0.0,-0.7,12.9,330,100,0.0,0.3
3,37.67,2022-01-01 03:00:00,0.0,-0.2,14.4,1,100,0.0,0.0
4,39.70,2022-01-01 04:00:00,0.0,-0.4,12.2,360,100,0.0,0.0


In [28]:
merged.to_csv("../data/processed/sweden_merged.csv", index=False)